# Lab 3a — Local Post-hoc Explainability on Structured Data


This lab covers **LIME (Local Interpretable Model-agnostic Explanations)**, a local post-hoc explainability technique applied to a black-box classifier on the Titanic dataset.

| Concept | Description |
|---|---|
| **Local surrogate** | An interpretable model that approximates the black-box *near a specific instance* |
| **Perturbation** | Slightly modified copies of the instance used to probe the black-box |
| **Proximity weighting** | Perturbed samples closer to the instance of interest are given higher weight |

**Guide:**
- ⚙️ Data preprocessing (Titanic + LIME-specific encoding)
- 🔍 LIME explainer setup
- 🎲 Exploring LIME parameters


## 🔍 What is LIME?

LIME is a **local surrogate model**. It tests **what happens to the predictions** when you give **variations of your data** into the machine learning model.

The main steps are:

1. LIME generates **a new dataset** consisting of **perturbed samples** and the corresponding **predictions** of the black-box model.
2. On this new dataset → LIME trains an **interpretable model** (weighted by the proximity of the sampled instances to the instance of interest).
3. The learned model should be a **good local approximation** of the machine learning model predictions, but it does **not** have to be a good global approximation.

> 💡 The key insight: we only need the **prediction function** of the black-box — not its internals.


## ⚙️ Data Preprocessing (~15 min)

We use the **[Titanic](https://www.openml.org/search?type=data&sort=runs&id=40945&status=active)** dataset loaded from OpenML.
The preprocessing pipeline for LIME is slightly different from previous labs: **categorical features are Label-Encoded first** (for the explainer), then additionally One-Hot Encoded (for the classifier).

> 💡 Apply the same good habits from Lab 2: always fit transformations on the **training set only**, then apply them to the test set!


### ⚙️ Step 1 — Imports & Load

In [1]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

import pandas as pd
import numpy as np

# Import the required libraries for this exercise
### Write your code here!

print("✅ Libraries loaded")

✅ Libraries loaded


### ⚙️ Step 2 — Load the Titanic Dataset

The [Titanic dataset](https://www.openml.org/search?type=data&sort=runs&id=40945&status=active) contains passenger information and whether they survived.
We want to predict the column **`survived`** (0 = no, 1 = yes).


In [2]:
# # Load input features and target variable
# df, y = fetch_openml("titanic", version=1, as_frame=True, parser='auto', return_X_y=True)

# # The "survived" column contains the target variable
# df["survived"] = y
# Load Titanic (same source/columns as OpenML "titanic" v1) from a direct CSV
url = "https://hbiostat.org/data/repo/titanic3.csv"
df = pd.read_csv(url, na_values="?")

# The "survived" column already contains the target variable
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (1309, 14)


,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.00,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.92,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.00,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.00,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.00,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


### ⚙️ Step 3 — Stratified Train / Test Split

> 🔑 **Key reminder:** split *before* any transformation. Fitting scalers or imputers on the full dataset causes **data leakage**.
Use an **80/20** ratio, **shuffle** the dataset, and **stratify** by the target variable.


In [3]:
#### START CODE HERE ####
# Split 80% train / 20% test, shuffle=True, stratify by 'survived'
df_train, df_test = train_test_split(df, test_size=0.2, shuffle=True, random_state=42, stratify=df['survived'])

#### END CODE HERE ####

### ⚙️ Step 4 — Handle Missing Values

> 🔑 **Key reminder:** compute statistics (mean, median, mode) on the **training set only**, then apply to both sets.

Fill missing values as follows:
- `age` → training **mean**
- `fare` → training **median**
- `embarked` → training **most frequent** value


In [4]:
#### START CODE HERE ####
# Fill 'age' NaN with the TRAINING mean
print(f'Number of null values in Train before pre-processing: {df_train.age.isnull().sum()}/{len(df_train)}')
print(f'Number of null values in Test before pre-processing: {df_test.age.isnull().sum()}/{len(df_test)}')

df_train['age'] = df_train['age'].fillna(df_train['age'].mean())
df_test['age'] = df_test['age'].fillna(df_train['age'].mean())

print(f'Number of null values in Train after pre-processing: {df_train.age.isnull().sum()}/{len(df_train)}')
print(f'Number of null values in Test after pre-processing: {df_test.age.isnull().sum()}/{len(df_test)}')

# Fill 'fare' NaN with the TRAINING median
print(f'Number of null values in Train before pre-processing: {df_train.fare.isnull().sum()}/{len(df_train)}')
print(f'Number of null values in Test before pre-processing: {df_test.fare.isnull().sum()}/{len(df_test)}')

df_train['fare'] = df_train['fare'].fillna(df_train['fare'].median())
df_test['fare'] = df_test['fare'].fillna(df_train['fare'].median())

print(f'Number of null values in Train after pre-processing: {df_train.fare.isnull().sum()}/{len(df_train)}')
print(f'Number of null values in Test after pre-processing: {df_test.fare.isnull().sum()}/{len(df_test)}')

# Fill 'embarked' NaN with the TRAINING most frequent value
print(f'Number of null values in Train before pre-processing: {df_train.embarked.isnull().sum()}/{len(df_train)}')
print(f'Number of null values in Test before pre-processing: {df_test.embarked.isnull().sum()}/{len(df_test)}')

imp = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
df_train[['embarked']] = imp.fit_transform(df_train[['embarked']])
df_test[['embarked']] = imp.transform(df_test[['embarked']])

print(f'Number of null values in Train after pre-processing: {df_train.embarked.isnull().sum()}/{len(df_train)}')
print(f'Number of null values in Test after pre-processing: {df_test.embarked.isnull().sum()}/{len(df_test)}')

#### END CODE HERE ####

Number of null values in Train before pre-processing: 209/1047
Number of null values in Test before pre-processing: 54/262
Number of null values in Train after pre-processing: 0/1047
Number of null values in Test after pre-processing: 0/262
Number of null values in Train before pre-processing: 1/1047
Number of null values in Test before pre-processing: 0/262
Number of null values in Train after pre-processing: 0/1047
Number of null values in Test after pre-processing: 0/262
Number of null values in Train before pre-processing: 0/1047
Number of null values in Test before pre-processing: 2/262
Number of null values in Train after pre-processing: 0/1047
Number of null values in Test after pre-processing: 0/262


### ⚙️ Step 5 — Drop Uninformative & Leaky Columns

Remove columns that are either **not informative** for the task, or that **leak information** about the target variable (`survived`):
- Not informative: `name`, `ticket`
- Leaky (contain post-hoc survival info): `cabin`, `body`, `boat`, `home.dest`


In [5]:
#### START CODE HERE ####
# Drop: 'name', 'ticket', 'cabin', 'body', 'boat', 'home.dest'

df_train = df_train.drop(columns=['name','ticket','cabin', 'body', 'boat', 'home.dest'])
df_test = df_test.drop(columns=['name','ticket','cabin', 'body', 'boat', 'home.dest'])
#### END CODE HERE ####

### ⚙️ Step 6 — Extract Features and Target

Separate the input features `X` and the target variable `y` for both the training and test sets.


In [6]:
#### START CODE HERE ####
# Extract X_train, X_test, y_train, y_test

y_train = df_train['survived']               # Target variable training set
X_train = df_train.drop('survived', axis=1)  # Features training set

# Extract target variable and input features for the testing data
y_test = df_test['survived']                 # Target variable test set
X_test = df_test.drop('survived', axis=1)    # Features test set

#### END CODE HERE ####

### ⚙️ Step 7 — LIME-Specific Encoding

The LIME explainer (and most classifiers) requires **numerical data**, even for categorical features.
We apply a **two-stage encoding** strategy:

| Stage | Encoder | Used by |
|---|---|---|
| **Stage 1** | `LabelEncoder` per categorical column | LIME explainer |
| **Stage 2** | `OneHotEncoder` + `MinMaxScaler` via `ColumnTransformer` | Random Forest classifier |

> 🔑 **Key reason:** the explainer must ensure a categorical feature only takes **one value** at a time. LabelEncoding achieves this, while OHE would split it across multiple binary columns.

We also save a `categorical_names` dictionary mapping each feature index to the list of its original string labels — this allows LIME to display readable feature values in the explanation.


#### Step 7a — Identify Categorical Columns

Identify the **categorical columns** (dtype `category` or `object`) and save their **column indices** — LIME requires indices, not names.
Both `category` and `object` dtypes represent categorical columns here.


In [13]:
# Display .info() to inspect dtypes
### Write your code here!
X_train.info()
X_test.info()
#y_train.info()
#colonnes sex et embarked à changer

<class 'pandas.core.frame.DataFrame'>
Index: 1047 entries, 999 to 668
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pclass    1047 non-null   int64  
 1   sex       1047 non-null   object 
 2   age       1047 non-null   float64
 3   sibsp     1047 non-null   int64  
 4   parch     1047 non-null   int64  
 5   fare      1047 non-null   float64
 6   embarked  1047 non-null   object 
dtypes: float64(2), int64(3), object(2)
memory usage: 65.4+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 262 entries, 1028 to 203
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pclass    262 non-null    int64  
 1   sex       262 non-null    object 
 2   age       262 non-null    float64
 3   sibsp     262 non-null    int64  
 4   parch     262 non-null    int64  
 5   fare      262 non-null    float64
 6   embarked  262 non-null    object 
dtypes: float64(2), int64(3), o

In [16]:
#### START CODE HERE ####
# Identify categorical columns and save their indices
# categorical_cols should be a list of integer indices
categorical_cols = [1,6]
#### END CODE HERE ####

#### Step 7b — LabelEncode for LIME

For each categorical feature:
1. Instantiate a `LabelEncoder()` and **fit+transform** on the **training set**.
2. Save the mapping: `categorical_names[feature_index] = le.classes_`
3. Save the encoder: `le_dict[feature_index] = le`
4. For the **test set**, apply only `.transform()` using the saved encoder.


In [ ]:
categorical_names = {}
le_dict = {}

#### START CODE HERE ####
# Fit LabelEncoder on train, transform train; then transform test
for feature in categorical_cols:
    ## Continue with your code here
    pass


#### END CODE HERE ####

print("categorical_names:", categorical_names)

In [ ]:
#### START CODE HERE ####
# Transform the test categorical columns using ONLY the encoders fitted on X_train.
for feature in categorical_cols:
    # TODO: retrieve the fitted LabelEncoder from le_dict and transform X_test
    pass
#### END CODE HERE ####


#### Step 7c — OneHotEncoder + MinMaxScaler for the Classifier

The **classifier** uses a representation different from the one shown to LIME:

- categorical features are **One-Hot Encoded (OHE)** so that category codes are not treated as ordered numbers;
- numerical features are **MinMax-scaled**;
- preprocessing parameters must be fitted on the **training set only** and reused on the test set.

> ⚠️ LIME still works with the seven original semantic features. Later, `predict_fn` will convert LIME's perturbed samples into the classifier representation.


In [ ]:
#### START CODE HERE ####
# Feature order:
# 0 pclass | 1 sex | 2 age | 3 sibsp | 4 parch | 5 fare | 6 embarked
# TODO: define the numerical column indices (categorical_cols was defined above)


# TODO: initialize OneHotEncoder(handle_unknown="ignore")


# TODO: initialize MinMaxScaler()


# TODO: create a ColumnTransformer that OHE-encodes categorical_cols
#       and MinMax-scales numerical_columns


# TODO: fit the transformer on X_train and transform X_train


# TODO: transform X_test using the already fitted transformer


#### END CODE HERE ####


> ✅ **Preprocessing complete.** Remember the LIME-specific rules:
> 1. Split *before* fitting any transformer.
> 2. LabelEncode for the **explainer**, OHE+scale for the **classifier**.
> 3. Always fit encoders on **train only**, then apply to both sets.


## Exercise 1 — Train the Black-Box Classifier 🌲

We train a **RandomForestClassifier** as our black-box model.

**Your tasks:**
1. Fit a `RandomForestClassifier` with `n_estimators=500` and `random_state=42`.
2. Calculate predictions with `.predict()`.
3. Calculate the `accuracy_score()`.

---

### 1.1 — Fit & Evaluate


In [ ]:
#### START CODE HERE ####
# TODO: fit RandomForestClassifier(n_estimators=500, random_state=42)

#### END CODE HERE ####


In [ ]:
#### START CODE HERE ####
# Calculate y_pred with .predict()

#### END CODE HERE ####

In [ ]:
#### START CODE HERE ####
# Calculate and print accuracy_score

#### END CODE HERE ####

## Exercise 2 — Explaining Predictions with LIME 🔍

Now we explain individual predictions using **LIME**.

**Key steps:**
1. Define a reproducibility constant (`RANDOM_STATE = 42`).
2. Instantiate `lime_tabular.LimeTabularExplainer` with `random_state=RANDOM_STATE`.
3. Define a **custom prediction function** `predict_fn` that converts LIME's seven semantic features into the OHE/scaled representation expected by the classifier.
4. Explain instance `i=0` using `num_samples=1000`.

> 💡 *What is the predicted class? Which features locally support or oppose that prediction?*

---

### 2.1 — Install & Import


In [ ]:
!pip install lime
from lime import lime_tabular

### 2.2 — Reproducibility


In [ ]:
#### START CODE HERE ####
# TODO: define RANDOM_STATE = 42 and set NumPy's seed as well

#### END CODE HERE ####


### 2.3 — Instantiate the LimeTabularExplainer

In [ ]:
#### START CODE HERE ####
# TODO: instantiate LimeTabularExplainer using:
# - X_train.values
# - mode='classification'
# - class_names=['not survived', 'survived']
# - feature_names=X_train.columns.tolist()
# - categorical_features=categorical_cols
# - categorical_names=categorical_names
# - kernel_width=3
# - random_state=RANDOM_STATE

#### END CODE HERE ####


### 2.4 — Define the Custom Prediction Function

In [ ]:
def predict_fn(x):
    """Convert LIME's semantic feature representation to the classifier representation."""
    # TODO:
    # 1. build a DataFrame from x with columns=X_train.columns
    # 2. apply ct.transform(...)
    # 3. return rf.predict_proba(...).astype(float)
    pass


### 2.5 — Explain Instance `i=0`

In [ ]:
#### START CODE HERE ####
# TODO: explain test instance i=0 with num_samples=1000 and display the explanation

#### END CODE HERE ####


## Exercise 3 — Exploring LIME Parameters 🎲

Now explore how LIME's sampling and display parameters affect the explanation.

**Your tasks:**
1. Instantiate a new `LimeTabularExplainer` with the **same configuration** as before, but deliberately omit `random_state`.
2. Explain the same instance (`i=1`) **five times** using the same `predict_fn`.
   > *Do the explanations change? Why?*
3. Compare `num_samples = 10`, `100`, and `1000`.
   > *How does the neighborhood size affect stability/fidelity and computation time?*
4. Vary `num_features` between 1 and 6.
   > *What changes in the displayed explanation?*
5. Compare the default Euclidean distance with another metric such as Manhattan distance.
   > *Where does this distance enter the LIME algorithm?*

---

### 3.1 — Repeated Explanations (Stability)


In [ ]:
#### START CODE HERE ####
# TODO: instantiate an UNSEEDED LimeTabularExplainer with the same
#       configuration as the baseline explainer (including kernel_width=3)

#### END CODE HERE ####


In [ ]:
#### START CODE HERE ####
# TODO: explain the same instance i=1 five times with the unseeded explainer.
#       Use the same predict_fn and, for example, num_samples=100.
#       Print exp_run.as_list() for easy comparison across runs.

#### END CODE HERE ####


### 3.2 — Effect of `num_samples`

`num_samples` controls how many synthetic perturbed observations are used to fit the local surrogate. Compare several values and observe both explanation stability and runtime.


In [ ]:
#### START CODE HERE ####
# TODO: for instance i=1, compare num_samples = 10, 100, and 1000.
# For a fair reproducibility comparison, recreate a seeded explainer
# (random_state=RANDOM_STATE) for each value before calling explain_instance.

#### END CODE HERE ####


### 3.3 — Effect of `num_features`

In [ ]:
#### START CODE HERE ####
# TODO: vary num_features (for example 1, 3, 6) while keeping num_samples=1000.
# Print exp.as_list() so you can see which terms are displayed.

#### END CODE HERE ####


### 3.4 — Effect of `distance_metric`

In [ ]:
#### START CODE HERE ####
# TODO: explain instance i=1 using num_samples=1000 and
# distance_metric='manhattan'. Compare it with the default Euclidean result.

#### END CODE HERE ####


### 🔍 Reflection Questions

> **Recall:** LIME generates a synthetic neighborhood by perturbing the instance. The selected `distance_metric` is then used to measure the distance between each perturbed observation and the original instance. LIME's kernel converts those distances into locality weights for fitting the surrogate. Perturbation generation and distance-based weighting are therefore distinct steps.

1. **Stability:** Why did running LIME multiple times without fixing the seed produce different explanations? What does this tell you about the reliability of local explanations?

2. **`num_samples`:** How does increasing `num_samples` affect the quality and stability of the local surrogate model? What is the trade-off with computation time? Does increasing `num_samples` make the explanation global?

3. **LabelEncoder vs OneHotEncoder:** Why does the LIME explainer work on LabelEncoded semantic features while the classifier uses OHE data? What problems could arise if the OHE representation itself were treated as the explainer's feature space?

4. **Local vs Global:** LIME provides a *local* explanation. Can it mislead you about the model's *global* behaviour? Give an example using the Titanic context (e.g. a first-class woman vs a third-class man).
